# NLP Lab Complete

All lab experiments with example outputs included.

## Preprocessing

In [ ]:
# ============================================================
# 01 - Reading data from files and preprocessing
# ============================================================

import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# ------------------------------------------------------------
# ------------------------------------------------------------
file_path = "data/sample.txt"

# Read text file
with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

print("Original text:")
print(text)

# Lowercase
text = text.lower()

# Remove punctuation
text = text.translate(str.maketrans("", "", string.punctuation))

# Tokenize
tokens = word_tokenize(text)

# Remove stopwords
stop_words = set(stopwords.words("english"))
tokens = [word for word in tokens if word not in stop_words]

print("\nProcessed tokens:")
print(tokens)

# Optional: stemming
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(word) for word in tokens]

print("\nStemmed tokens:")
print(stemmed_tokens)

# ============================================================
# ============================================================
# file_path = "data/your_file.txt"
#
# The rest of the program can remain the same.


Original text:
I like mutton biryani.
Chicken curry tastes good with mutton fry.
I enjoy grilled chicken.
Fish fry is tasty.
I am a non vegetarian foodie.

Processed tokens:
['like', 'mutton', 'biryani', 'chicken', 'curry', 'tastes', 'good', 'mutton', 'fry', 'enjoy', 'grilled', 'chicken', 'fish', 'fry', 'tasty', 'non', 'vegetarian', 'foodie']

Stemmed tokens:
['like', 'mutton', 'biriyani', 'chicken', 'curri', 'tast', 'good', 'mutton', 'fri', 'enjoy', 'grill', 'chicken', 'fish', 'fri', 'tasti', 'non', 'vegetarian', 'foodi']


## Word Embeddings

In [ ]:
# ============================================================
# 02 - Word Embeddings using Word2Vec
# ============================================================

# Install once if required:
# !pip install gensim

from gensim.models import Word2Vec

# ------------------------------------------------------------
# ------------------------------------------------------------
sentences = [
    ["i", "like", "mutton", "biryani"],
    ["mutton", "biryani", "is", "tasty"],
    ["i", "enjoy", "chicken", "curry"],
    ["chicken", "curry", "tastes", "good"],
    ["fish", "fry", "is", "tasty"],
    ["i", "like", "chicken"],
]

# Train Word2Vec
model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=2,
    min_count=1,
    workers=1,
    sg=1
)

print("Vocabulary:")
print(list(model.wv.index_to_key))

word = "chicken"

print("\nVector for:", word)
print(model.wv[word])

print("\nSimilarity between chicken and curry:")
print(model.wv.similarity("chicken", "curry"))

print("\nMost similar words to chicken:")
print(model.wv.most_similar("chicken", topn=3))

# ============================================================
# ============================================================
# sentences = [...]
#
# If you query a word, make sure that word occurs in the
# training corpus. Otherwise KeyError will occur.
#
# Important parameters:
# vector_size = number of dimensions
# window = context window size
# min_count = minimum word frequency
# sg=1 -> Skip-gram
# sg=0 -> CBOW


Vocabulary:
['chicken', 'curry', 'enjoy', 'fish', 'fry', 'good', 'i', 'is', 'like', 'mutton', 'tasty', 'tastes']

Vector for: chicken
[... 50-dimensional Word2Vec vector ...]

Similarity between chicken and curry:
0.99...

Most similar words to chicken:
[('curry', 0.99...), ('i', 0.98...), ('tastes', 0.97...)]


## TF-IDF + LSI

In [ ]:
# ============================================================
# 03 - TF-IDF + LSI
# ============================================================

import nltk
import pandas as pd
import string
import math
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# ============================================================
# DOCUMENTS
# ============================================================
documents = [
    "I like mutton biryani.",
    "Chicken curry tastes good with mutton fry.",
    "I enjoy grilled chicken.",
    "Fish fry is tasty.",
    "I am a non vegetarian foodie."
]

# ============================================================
# PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))
processed_docs = []

for doc in documents:
    doc = doc.lower()
    doc = doc.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(doc)
    tokens = [w for w in tokens if w not in stop_words]
    processed_docs.append(tokens)

print("Processed Documents\n")
for i, doc in enumerate(processed_docs):
    print(f"D{i+1}: {' '.join(doc)}")

# ============================================================
# VOCABULARY
# ============================================================

vocab = sorted(set(word for doc in processed_docs for word in doc))

print("\nVocabulary:\n")
print(vocab)

# ============================================================
# TF TABLE
# TF = count(word in document) / total words in document
# ============================================================

tf_table = pd.DataFrame(
    0.0,
    index=vocab,
    columns=[f"D{i+1}" for i in range(len(processed_docs))]
)

for i, doc in enumerate(processed_docs):
    total = len(doc)

    for word in vocab:
        tf = doc.count(word) / total
        tf_table.loc[word, f"D{i+1}"] = round(tf, 3)

print("\n====================")
print("TERM FREQUENCY (TF)")
print("====================\n")
print(tf_table)

# ============================================================
# IDF TABLE
# IDF = log(N / DF)
# ============================================================

N = len(processed_docs)

idf_values = {}

for word in vocab:
    df = sum(word in doc for doc in processed_docs)
    idf = math.log(N / df)
    idf_values[word] = round(idf, 3)

idf_table = pd.DataFrame.from_dict(
    idf_values,
    orient="index",
    columns=["IDF"]
)

print("\n====================")
print("IDF TABLE")
print("====================\n")
print(idf_table)

# ============================================================
# TF-IDF TABLE
# ============================================================

tfidf_table = tf_table.copy()

for word in vocab:
    tfidf_table.loc[word] = (
        tf_table.loc[word] * idf_values[word]
    ).round(3)

print("\n====================")
print("TF-IDF = TF × IDF")
print("====================\n")
print(tfidf_table)

# ============================================================
# TF-IDF MATRIX
# sklearn expects:
# rows = documents
# columns = terms
# ============================================================

tfidf_matrix = tfidf_table.T.values

print("\n====================")
print("TF-IDF MATRIX")
print("====================\n")
print(tfidf_matrix)

# ============================================================
# LSI USING TRUNCATED SVD
# ============================================================

# Number of latent dimensions.
requested_components = 2

# TruncatedSVD cannot use as many components as the number
# of features in some small datasets.
max_components = min(tfidf_matrix.shape[0], tfidf_matrix.shape[1] - 1)

if max_components < 1:
    print("\nNot enough data/features for TruncatedSVD.")
else:
    n_components = min(requested_components, max_components)

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=42
    )

    document_lsi = svd.fit_transform(tfidf_matrix)

    print("\n====================")
    print("LSI COMPONENTS")
    print("====================\n")
    print(svd.components_)

    print("\n====================")
    print("DOCUMENT REPRESENTATION IN LSI SPACE")
    print("====================\n")

    lsi_table = pd.DataFrame(
        document_lsi,
        index=[f"D{i+1}" for i in range(len(documents))],
        columns=[f"Topic{i+1}" for i in range(n_components)]
    )
    print(lsi_table)

    print("\nExplained variance ratio:")
    print(svd.explained_variance_ratio_)

    # ========================================================
    # QUERY IN THE SAME LSI SPACE
    # ========================================================

    query = "mutton chicken fry"

    query_tokens = word_tokenize(query.lower())
    query_tokens = [
        w for w in query_tokens
        if w not in stop_words and w in vocab
    ]

    query_tf = []
    for word in vocab:
        query_tf.append(
            query_tokens.count(word) / len(query_tokens)
            if len(query_tokens) > 0 else 0
        )

    query_tfidf = [
        query_tf[i] * idf_values[word]
        for i, word in enumerate(vocab)
    ]

    query_lsi = svd.transform([query_tfidf])

    print("\nQuery:", query)
    print("Query LSI representation:")
    print(query_lsi)

    similarities = cosine_similarity(query_lsi, document_lsi)[0]

    print("\nCosine similarity with documents:")
    for i, score in enumerate(similarities):
        print(f"D{i+1}: {score:.3f}")

# ============================================================
# PIPELINE TO REMEMBER
# ============================================================
# Documents
#     ↓
# Lowercase
#     ↓
# Remove punctuation
#     ↓
# Tokenize
#     ↓
# Remove stopwords
#     ↓
# Vocabulary
#     ↓
# TF
#     ↓
# IDF
#     ↓
# TF-IDF
#     ↓
# LSI / TruncatedSVD
#
# IMPORTANT:
# tfidf_table is [terms x documents]
# sklearn SVD input here is [documents x terms],
# therefore we use tfidf_table.T

# ============================================================
# ============================================================
# documents = [...]
#
# requested_components = 2
#
# query = "..."


Processed Documents

D1: like mutton biryani
D2: chicken curry tastes good mutton fry
D3: enjoy grilled chicken
D4: fish fry tasty
D5: non vegetarian foodie

Vocabulary:

['biryani', 'chicken', 'curry', 'enjoy', 'fish', 'foodie', 'fry', 'good', 'grilled', 'like', 'mutton', 'non', 'tastes', 'tasty', 'vegetarian']

TERM FREQUENCY (TF)

                          D1            D2            D3            D4            D5
       biryani         0.333         0.000         0.000         0.000         0.000
       chicken         0.000         0.167         0.333         0.000         0.000
         curry         0.000         0.167         0.000         0.000         0.000
         enjoy         0.000         0.000         0.333         0.000         0.000
          fish         0.000         0.000         0.000         0.333         0.000
        foodie         0.000         0.000         0.000         0.000         0.333
           fry         0.000         0.167         0.000         0.33

## N-Grams

In [ ]:
# ============================================================
# 04 - N-GRAMS
# ============================================================

import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("punkt")
nltk.download("punkt_tab")

text = "I like mutton biryani and chicken curry."

tokens = word_tokenize(text.lower())

# ------------------------------------------------------------
# n=1 -> unigram
# n=2 -> bigram
# n=3 -> trigram
# ------------------------------------------------------------

n = 2

ngram_list = list(ngrams(tokens, n))
counts = Counter(ngram_list)

print(f"{n}-grams:")
for item, count in counts.items():
    print(item, ":", count)

# Show all three separately
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print("\nUnigrams:")
print(Counter(unigrams))

print("\nBigrams:")
print(Counter(bigrams))

print("\nTrigrams:")
print(Counter(trigrams))

# ============================================================
# SCIKIT-LEARN VERSION
# ============================================================

documents = [
    "I like mutton biryani",
    "I like chicken curry",
    "mutton fry is tasty"
]

vectorizer = CountVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(documents)

print("\nScikit-learn n-grams:")
print(vectorizer.get_feature_names_out())
print(X.toarray())

# ngram_range=(1,1) -> unigrams only
# ngram_range=(2,2) -> bigrams only
# ngram_range=(3,3) -> trigrams only
# ngram_range=(1,3) -> unigrams + bigrams + trigrams

# ============================================================
# ============================================================
# text = "..."
#
# For NLTK:
# n = 1 / 2 / 3
#
# For sklearn:
# ngram_range=(1,2), etc.


2-grams:
('i', 'like') : 1
('like', 'mutton') : 1
('mutton', 'biryani') : 1
('biryani', 'and') : 1
('and', 'chicken') : 1
('chicken', 'curry') : 1

Unigrams:
Counter({('i',): 1, ('like',): 1, ('mutton',): 1, ('biryani',): 1, ('and',): 1, ('chicken',): 1, ('curry',): 1})

Bigrams:
Counter({('i', 'like'): 1, ('like', 'mutton'): 1, ('mutton', 'biryani'): 1, ('biryani', 'and'): 1, ('and', 'chicken'): 1, ('chicken', 'curry'): 1})

Trigrams:
Counter({('i', 'like', 'mutton'): 1, ('like', 'mutton', 'biryani'): 1, ('mutton', 'biryani', 'and'): 1, ('biryani', 'and', 'chicken'): 1, ('and', 'chicken', 'curry'): 1})

Scikit-learn n-grams:
['chicken' 'chicken curry' 'curry' 'i' 'i like' 'is' 'is tasty' 'like' 'like chicken' 'like mutton' 'mutton' 'mutton fry' 'tasty']


## HMM POS Tagging

In [ ]:
# ============================================================
# 05 - POS TAGGING USING HMMLEARN
# ============================================================

# Install once if required:
# !pip install hmmlearn

import numpy as np

try:
    from hmmlearn.hmm import CategoricalHMM
except ImportError:
    # Older hmmlearn versions may expose MultinomialHMM.
    # For this lab template, install a recent hmmlearn version.
    raise ImportError("Install/update hmmlearn so CategoricalHMM is available.")

# ============================================================
# STATES = POS TAGS
# ============================================================

states = ["NN", "VB", "RB"]
state2idx = {state: i for i, state in enumerate(states)}

# ============================================================
# SMALL MANUALLY SPECIFIED VOCABULARY
# ============================================================

vocab = [
    "<UNK>",
    "dogs",
    "cats",
    "run",
    "chase",
    "quickly",
    "slowly"
]

word2idx = {word: i for i, word in enumerate(vocab)}

# ============================================================
# START PROBABILITIES
# P(first POS tag)
# ============================================================

startprob = np.array([
    0.7,  # NN
    0.2,  # VB
    0.1   # RB
])

# ============================================================
# TRANSITION PROBABILITIES
# P(current tag | previous tag)
# Rows = previous state
# Columns = current state
# ============================================================

transmat = np.array([
    [0.1, 0.7, 0.2],  # NN -> NN, VB, RB
    [0.6, 0.1, 0.3],  # VB -> NN, VB, RB
    [0.5, 0.2, 0.3],  # RB -> NN, VB, RB
])

# ============================================================
# EMISSION PROBABILITIES
# P(word | POS tag)
# Rows = POS states
# Columns = vocabulary
# ============================================================

emissionprob = np.array([
    # <UNK> dogs cats run chase quickly slowly
    [0.02, 0.35, 0.35, 0.03, 0.05, 0.10, 0.10],  # NN
    [0.02, 0.03, 0.03, 0.40, 0.40, 0.10, 0.02],  # VB
    [0.02, 0.02, 0.02, 0.02, 0.02, 0.45, 0.45],  # RB
])

# ============================================================
# CREATE CATEGORICAL HMM
# ============================================================

model = CategoricalHMM(
    n_components=len(states),
    init_params=""
)

model.startprob_ = startprob
model.transmat_ = transmat
model.emissionprob_ = emissionprob

# ============================================================
# TAG A SENTENCE
# ============================================================

def hmm_tag_sentence(words):
    # Unknown words become <UNK>
    obs = [
        word2idx.get(word.lower(), word2idx["<UNK>"])
        for word in words
    ]

    X = np.array(obs).reshape(-1, 1)

    log_probability, state_sequence = model.decode(
        X,
        algorithm="viterbi"
    )

    tags = [states[i] for i in state_sequence]

    return list(zip(words, tags)), log_probability


test_sentence = ["dogs", "run", "quickly"]

result, log_probability = hmm_tag_sentence(test_sentence)

print("Input:")
print(test_sentence)

print("\nPOS tags:")
print(result)

print("\nViterbi log probability:")
print(log_probability)

# ============================================================
# WHAT EACH HMM PART MEANS
# ============================================================
# states          -> hidden POS tags
# observations    -> words
# startprob_      -> probability of first POS tag
# transmat_       -> transition probabilities between tags
# emissionprob_   -> probability of a word given a tag
# decode(...viterbi)
#                 -> finds most likely hidden POS sequence

# ============================================================
# ============================================================
# states
# vocab
# startprob
# transmat
# emissionprob
# test_sentence
#
# If a test word is not in vocab, it is automatically mapped
# to <UNK>, so the code does not crash.


Input:
['dogs', 'run', 'quickly']

POS tags:
[('dogs', 'NN'), ('run', 'VB'), ('quickly', 'RB')]

Viterbi log probability:
-5.8...


## HMM Viterbi From Scratch

In [ ]:
# ============================================================
# 06 - HMM VITERBI FROM SCRATCH
# ============================================================

import math

# States = POS tags
states = ["NN", "VB", "RB"]

# Start probabilities
start_prob = {
    "NN": 0.7,
    "VB": 0.2,
    "RB": 0.1
}

# Transition probabilities
transition_prob = {
    "NN": {"NN": 0.1, "VB": 0.7, "RB": 0.2},
    "VB": {"NN": 0.6, "VB": 0.1, "RB": 0.3},
    "RB": {"NN": 0.5, "VB": 0.2, "RB": 0.3}
}

# Emission probabilities
emission_prob = {
    "NN": {
        "dogs": 0.35,
        "cats": 0.35,
        "run": 0.03,
        "chase": 0.05,
        "quickly": 0.10,
        "slowly": 0.10
    },
    "VB": {
        "dogs": 0.03,
        "cats": 0.03,
        "run": 0.40,
        "chase": 0.40,
        "quickly": 0.10,
        "slowly": 0.02
    },
    "RB": {
        "dogs": 0.02,
        "cats": 0.02,
        "run": 0.02,
        "chase": 0.02,
        "quickly": 0.45,
        "slowly": 0.45
    }
}

# Small probability for an unknown word
UNKNOWN_PROB = 0.02

def emit(tag, word):
    return emission_prob[tag].get(word.lower(), UNKNOWN_PROB)

def viterbi(words):
    # dp[t][state] = best probability/log-probability
    # ending in 'state' at position t
    dp = []
    backpointer = []

    # --------------------------------------------------------
    # 1. INITIALIZATION
    # --------------------------------------------------------
    first = words[0]
    first_scores = {}
    first_back = {}

    for state in states:
        probability = start_prob[state] * emit(state, first)
        first_scores[state] = math.log(probability)
        first_back[state] = None

    dp.append(first_scores)
    backpointer.append(first_back)

    # --------------------------------------------------------
    # 2. RECURSION
    # --------------------------------------------------------
    for t in range(1, len(words)):
        word = words[t]
        scores = {}
        backs = {}

        for current_state in states:
            best_score = float("-inf")
            best_previous = None

            for previous_state in states:
                transition = transition_prob[previous_state][current_state]
                emission = emit(current_state, word)

                score = (
                    dp[t - 1][previous_state]
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:
                    best_score = score
                    best_previous = previous_state

            scores[current_state] = best_score
            backs[current_state] = best_previous

        dp.append(scores)
        backpointer.append(backs)

    # --------------------------------------------------------
    # 3. TERMINATION
    # --------------------------------------------------------
    last_state = max(dp[-1], key=dp[-1].get)
    best_score = dp[-1][last_state]

    # --------------------------------------------------------
    # 4. BACKTRACKING
    # --------------------------------------------------------
    best_path = [last_state]

    for t in range(len(words) - 1, 0, -1):
        previous_state = backpointer[t][best_path[-1]]
        best_path.append(previous_state)

    best_path.reverse()

    return best_path, best_score

# Test
sentence = ["dogs", "run", "quickly"]

tags, log_probability = viterbi(sentence)

print("Sentence:")
print(sentence)

print("\nMost likely POS tag sequence:")
print(tags)

print("\nLog probability:")
print(log_probability)

print("\nWord / Tag:")
for word, tag in zip(sentence, tags):
    print(word, "/", tag)

# ============================================================
# VITERBI STEPS TO REMEMBER
# ============================================================
# 1. Initialization
# 2. Recursion
# 3. Termination
# 4. Backtracking
#
# Formula:
# V_t(s) = max(previous_state)
#          V_(t-1)(previous_state)
#          * transition(previous_state, s)
#          * emission(s, word_t)
#
# We use log probabilities in the code to avoid multiplying
# many very small numbers.

# ============================================================
# ============================================================
# states
# start_prob
# transition_prob
# emission_prob
# sentence
#


Sentence:
['dogs', 'run', 'quickly']

Most likely POS tag sequence:
['NN', 'VB', 'RB']

Log probability:
-5.8...

Word / Tag:
dogs / NN
run / VB
quickly / RB


## CRF POS Tagging

In [ ]:
# ============================================================
# 07 - POS TAGGING USING CRF
# ============================================================

# Install once:
# !pip install sklearn-crfsuite

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.metrics import classification_report

# ============================================================
# SMALL TRAINING CORPUS
# ============================================================

train_sents = [
    [
        ("tiger", "NN"),
        ("chases", "VB"),
        ("deer", "NN")
    ],
    [
        ("civet", "NN"),
        ("chases", "VB"),
        ("deer", "NN")
    ],
    [
        ("civet", "NN"),
        ("makes", "VB"),
        ("loud", "RB")
    ],
    [
        ("dog", "NN"),
        ("runs", "VB"),
        ("quickly", "RB")
    ],
]

# ============================================================
# FEATURE EXTRACTION
# ============================================================

def word2features(sent, i):
    word = sent[i][0]
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:3]": word[:3],
        "word[:2]": word[:2],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "position": i,
        "is_first": i == 0,
        "is_last": i == len(sent) - 1,
    }

    if i > 0:
        previous_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": previous_word.lower(),
            "-1:word[-3:]": previous_word[-3:],
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word[-3:]": next_word[-3:],
        })
    else:
        features["EOS"] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for word, tag in sent]

def sent2tokens(sent):
    return [word for word, tag in sent]

# Convert training data
X_train = [sent2features(sent) for sent in train_sents]
y_train = [sent2labels(sent) for sent in train_sents]

# ============================================================
# TRAIN CRF
# ============================================================

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

# ============================================================
# PREDICT A NEW SENTENCE
# ============================================================

def crf_tag_sentence(words):
    # CRF feature extraction expects word/tag pairs,
    # but tags are placeholders because this is test input.
    sent = [(word, "NN") for word in words]

    features = sent2features(sent)
    predicted_tags = crf.predict_single(features)

    return list(zip(words, predicted_tags))

test_sentence = ["tiger", "chases", "deer"]

result = crf_tag_sentence(test_sentence)

print("Input:")
print(test_sentence)

print("\nCRF output:")
for word, tag in result:
    print(word, "/", tag)

# ============================================================
# EVALUATION
# ============================================================

y_pred = crf.predict(X_train)

print("\nClassification report:")
print(
    classification_report(
        [tag for sent in y_train for tag in sent],
        [tag for sent in y_pred for tag in sent],
        zero_division=0
    )
)

print("\nCRF flat accuracy:")
print(metrics.flat_accuracy_score(y_train, y_pred))

print("\nCRF flat F1:")
print(metrics.flat_f1_score(y_train, y_pred, average="weighted"))

# ============================================================
# ============================================================
# train_sents = [...]
#
# Test sentence:
# test_sentence = [...]
#
# IMPORTANT:
# For each training sentence, the number of words must equal
# the number of POS labels.
#
# Example:
# [("dog","NN"), ("runs","VB")]
# has 2 words and 2 labels.


Input:
['tiger', 'chases', 'deer']

CRF output:
tiger / NN
chases / VB
deer / NN

Classification report:
              precision    recall  f1-score   support

          NN       1.00      1.00      1.00         7
          RB       1.00      1.00      1.00         2
          VB       1.00      1.00      1.00         4

    accuracy                           1.00        13
   macro avg       1.00      1.00      1.00        13
weighted avg       1.00      1.00      1.00        13

CRF flat accuracy:
1.0

CRF flat F1:
1.0
